# 03 — Modelagem

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, roc_auc_score

RANDOM_STATE = 42
CAMINHO_DADOS = Path('..') / 'data' / 'raw' / 'WineQT.csv'
TARGET = 'qualidade_do_vinho'
pd.set_option('display.max_columns', None)
PROCESSED = Path('..') / 'data' / 'processed'
MODELS = Path('..') / 'results' / 'models'

In [ ]:
df = pd.read_csv(PROCESSED / "dataset_tratado.csv")

Para realizar a previsão da qualidade dos vinhos, escolhemos três modelos de classificação: Regressão Logística, Árvore de Decisão e Random Forest. A escolha permite comparar métodos com características diferentes.

Os modelos foram avaliados em três etapas, para verificar os resultados tanto com as configurações iniciais quanto após o ajuste dos parâmetros:

Modelos sem ajuste: treinamento com os dados originais e parâmetros padrão.

Modelos com GridSearchCV: busca pelas melhores combinações de parâmetros utilizando os dados originais.

Modelos com GridSearchCV e padronização: repetição do ajuste com o StandardScaler integrado a um Pipeline.

Ao final de cada etapa, avaliamos o desempenho dos modelos no conjunto de teste. Além da acurácia, analisamos a precisão (precision), o recall e o F1-score da classe Alto. Essas métricas são importantes porque a quantidade de vinhos em cada classe é diferente, o que significa que a acurácia, quando analisada sozinha, pode não representar adequadamente a capacidade do modelo de identificar vinhos de alta qualidade.

## 1. Split treino/teste

Separamos 20% dos vinhos para teste. O `stratify` mantém a mesma proporção de vinhos Alto no treino e no teste.

In [ ]:
X = df.drop(TARGET, axis=1)
y = df[TARGET]


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)

print(f'Treino: {X_train.shape[0]} amostras')
print(f'Teste: {X_test.shape[0]} amostras')

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

## 2. Modelos candidatos

Regressão Logística: é um modelo linear que estima a probabilidade de um vinho pertencer à classe Alto. Seus coeficientes ajudam a entender como cada variável está relacionada à previsão.

Árvore de Decisão: utiliza divisões baseadas nos valores das variáveis para classificar os vinhos. Isso permite representar o processo de decisão por meio de regras, tornando o resultado mais fácil de interpretar.

Random Forest: combina várias árvores de decisão para produzir uma classificação. Essa abordagem permite considerar relações mais complexas entre as características dos vinhos.

In [ ]:
modelos = {
    'Regressão Logística': LogisticRegression(random_state=RANDOM_STATE, class_weight='balanced', max_iter=5000),
    'Árvore de Decisão': DecisionTreeClassifier(random_state=RANDOM_STATE, class_weight='balanced'),
    'Random Forest': RandomForestClassifier(random_state=RANDOM_STATE, class_weight='balanced')
}

## 3. Validação cruzada e ajuste de parâmetros

### 4.1 Modelos sem ajuste (dados originais)

Primeiro treinamos os três modelos com os parâmetros padrão. Além do resultado no teste, calculamos o F1 com validação cruzada: o treino é dividido em 5 partes, o modelo treina em 4 e é avaliado na que sobrou, repetindo 5 vezes. Assim o resultado não depende de uma única divisão dos dados.

In [ ]:
resultado_base = []
for nome, modelo in modelos.items():
    modelo.fit(X_train, y_train)
    y_predito = modelo.predict(X_test)
    f1_cv = cross_val_score(modelo, X_train, y_train, cv=cv, scoring='f1').mean()
    resultado_base.append([nome,
                           accuracy_score(y_test, y_predito),
                           precision_score(y_test, y_predito),
                           recall_score(y_test, y_predito),
                           f1_score(y_test, y_predito),
                           f1_cv])

resultado_base = pd.DataFrame(resultado_base, columns=['Modelo', 'Taxa de acerto', 'Precisão', 'Recall', 'F1', 'F1 (validação cruzada)']).round(2)
resultado_base

Sem nenhum ajuste, o Random Forest teve a maior taxa de acerto no teste (0,91), mas encontrou só 1 de cada 3 vinhos Alto (recall de 0,33), e na validação cruzada teve o pior F1 (0,36). Ou seja, a taxa de acerto alta vem de classificar quase tudo como Baixo/Médio. A Regressão Logística teve o melhor F1 no teste (0,51) e na validação cruzada (0,49). A Árvore de Decisão sem limite de profundidade teve F1 de 0,40 na validação cruzada, porque cresce até decorar o treino.

### 4.2 Modelos com GridSearchCV (dados originais)

O GridSearchCV testa todas as combinações de parâmetros que passamos e escolhe a que tem o melhor F1 na validação cruzada, usando só os dados de treino.

In [ ]:
parametros = {
    'Regressão Logística': {
        'C': [0.001, 0.01, 0.1, 1, 10, 100]
    },
    'Árvore de Decisão': {
        'max_depth': [3, 4, 5, 6, 8, None],
        'min_samples_leaf': [1, 5, 10, 20],
        'criterion': ['gini', 'entropy']
    },
    'Random Forest': {
        'n_estimators': [100, 300],
        'max_depth': [None, 5, 10],
        'min_samples_leaf': [1, 3, 5],
        'criterion': ['gini', 'entropy']
    }
}

In [ ]:
resultado_grid = []
for nome, modelo in modelos.items():
    grid = GridSearchCV(modelo, parametros[nome], cv=cv, scoring='f1', n_jobs=-1)
    grid.fit(X_train, y_train)
    y_predito = grid.predict(X_test)
    print(nome, grid.best_params_)
    resultado_grid.append([nome,
                           accuracy_score(y_test, y_predito),
                           precision_score(y_test, y_predito),
                           recall_score(y_test, y_predito),
                           f1_score(y_test, y_predito),
                           grid.best_score_])

resultado_grid = pd.DataFrame(resultado_grid, columns=['Modelo', 'Taxa de acerto', 'Precisão', 'Recall', 'F1', 'F1 (validação cruzada)']).round(2)
resultado_grid

Com o GridSearchCV, o F1 na validação cruzada melhorou nos três modelos:
- Árvore de Decisão: de 0,40 para 0,49, limitando a profundidade em 5.
- Random Forest: de 0,36 para 0,55, com no mínimo 5 vinhos por folha.
- Regressão Logística: de 0,49 para 0,50, com C = 10.

No teste, o Random Forest ajustado encontrou bem mais vinhos Alto (recall de 0,33 para 0,67), mas também passou a marcar mais vinhos comuns como Alto, por isso a taxa de acerto caiu de 0,91 para 0,89. Como o teste tem só 27 vinhos Alto, confiamos mais na validação cruzada para escolher os parâmetros.

### 4.3 Modelos com GridSearchCV (dados padronizados)

Agora repetimos o GridSearchCV com os dados padronizados. O StandardScaler fica dentro de um Pipeline junto com o modelo, assim em cada parte da validação cruzada ele é ajustado só com os dados de treino daquela parte, e o teste nunca influencia a padronização.

In [ ]:
grids_padronizados = {}
resultado_grid_padronizado = []
for nome, modelo in modelos.items():
    pipeline = Pipeline([('scaler', StandardScaler()), ('modelo', modelo)])
    parametros_pipeline = {'modelo__' + chave: valores for chave, valores in parametros[nome].items()}
    grid = GridSearchCV(pipeline, parametros_pipeline, cv=cv, scoring='f1', n_jobs=-1)
    grid.fit(X_train, y_train)
    grids_padronizados[nome] = grid
    y_predito = grid.predict(X_test)
    print(nome, grid.best_params_)
    resultado_grid_padronizado.append([nome,
                                       accuracy_score(y_test, y_predito),
                                       precision_score(y_test, y_predito),
                                       recall_score(y_test, y_predito),
                                       f1_score(y_test, y_predito),
                                       grid.best_score_])

resultado_grid_padronizado = pd.DataFrame(resultado_grid_padronizado, columns=['Modelo', 'Taxa de acerto', 'Precisão', 'Recall', 'F1', 'F1 (validação cruzada)']).round(2)
resultado_grid_padronizado

A padronização só mudou de fato a Regressão Logística: o F1 na validação cruzada subiu de 0,50 para 0,52 e o melhor C mudou de 10 para 100. No teste o F1 dela ficou em 0,49, um vinho Alto a menos que na etapa anterior, o que está dentro da variação esperada com só 27 vinhos Alto. Isso era esperado, porque a Regressão Logística depende da escala das variáveis.

Na Árvore de Decisão o resultado foi idêntico, porque as árvores só comparam valores ("álcool maior que X") e a escala não altera essa comparação. No Random Forest o F1 na validação cruzada ficou praticamente igual (0,55 para 0,54). A combinação de parâmetros escolhida mudou porque várias combinações ficam quase empatadas, e uma diferença mínima basta para trocar a vencedora.

## 4. Comparação das três etapas

In [ ]:
comparacao_acerto = pd.DataFrame({
    'Sem ajuste': resultado_base.set_index('Modelo')['Taxa de acerto'],
    'GridSearch': resultado_grid.set_index('Modelo')['Taxa de acerto'],
    'GridSearch + padronizado': resultado_grid_padronizado.set_index('Modelo')['Taxa de acerto']
})

comparacao_f1 = pd.DataFrame({
    'Sem ajuste': resultado_base.set_index('Modelo')['F1'],
    'GridSearch': resultado_grid.set_index('Modelo')['F1'],
    'GridSearch + padronizado': resultado_grid_padronizado.set_index('Modelo')['F1']
})

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18,5))

comparacao_acerto.plot(kind='bar', rot=0, ax=axes[0])
axes[0].set_title('Taxa de acerto no teste')
axes[0].set_ylim(0, 1)

comparacao_f1.plot(kind='bar', rot=0, ax=axes[1])
axes[1].set_title('F1 da classe Alto no teste')
axes[1].set_ylim(0, 1)

for ax in axes:
    ax.set_xlabel('')
    ax.legend(loc='lower right')
    for container in ax.containers:
        ax.bar_label(container, fmt='%.2f')
plt.tight_layout()
plt.show()

A taxa de acerto muda pouco entre as etapas, e a base tem um certo desbalanceamento (87% Baixo/Médio no teste): um modelo que diz que nenhum vinho é de alta qualidade já acertaria 87%. O F1 da classe Alto mostra melhor a diferença: o GridSearch melhorou a Árvore de Decisão (0,44 para 0,52) e principalmente o Random Forest (0,49 para 0,61).

**Qual modelo venceu:** na validação cruzada da última etapa, o Random Forest teve F1 de 0,54, contra 0,52 da Regressão Logística e 0,49 da Árvore de Decisão. A vantagem sobre a Regressão Logística é pequena e pode estar dentro do ruído. No teste a margem é maior (0,59 contra 0,49 e 0,52), mas o teste tem só 27 vinhos Alto. A avaliação detalhada dos modelos finais está no notebook 04.

## 5. Salvar modelos

Salvamos os modelos da última etapa para que o notebook 04 avalie exatamente os mesmos modelos. Os arquivos ficam em `results/models/`, que não vai para o GitHub (`.gitignore`), porque podem ser gerados de novo rodando este notebook.

In [ ]:
import joblib

MODELS.mkdir(parents=True, exist_ok=True)
joblib.dump(grids_padronizados, MODELS / "grids_padronizados.joblib")